# Spotify Songs Report

This notebook analyzes a Spotify songs CSV file and generates two connected HTML pages:

- `spotify_home.html`
- `spotify_report.html`

Run each cell in order in Google Colab.


## Step 1: Upload the Spotify CSV file

When you run the next cell, choose your `spotify_songs.csv` file from your computer.


In [ ]:
from google.colab import files
uploaded = files.upload()

csv_file = list(uploaded.keys())[0]
print("Uploaded file:", csv_file)


## Step 2: Import packages and read the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
import shutil

df = pd.read_csv(csv_file)

print("Rows and columns:", df.shape)
df.head()


## Step 3: Clean column names and numeric values

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

for col in df.columns:
    if df[col].dtype == "object":
        cleaned = (
            df[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
        numeric = pd.to_numeric(cleaned, errors="coerce")
        if numeric.notna().sum() >= len(df) * 0.55:
            df[col] = numeric

df["streams_millions"] = df["streams"] / 1_000_000

df.head()


## Step 4: Create summary tables

In [ ]:
top_songs = (
    df[["song", "artist", "year", "streams_millions", "popularity"]]
    .sort_values("streams_millions", ascending=False)
    .head(10)
)

top_artists = (
    df.groupby("artist", as_index=False)["streams_millions"]
    .sum()
    .sort_values("streams_millions", ascending=False)
    .head(10)
)

genre_summary = (
    df.groupby("genre")
    .agg(
        avg_popularity=("popularity", "mean"),
        song_count=("song", "count"),
        avg_streams=("streams_millions", "mean")
    )
    .reset_index()
    .sort_values("avg_popularity", ascending=False)
    .head(10)
)

year_summary = (
    df.groupby("year")
    .agg(
        total_streams=("streams_millions", "sum"),
        avg_popularity=("popularity", "mean"),
        song_count=("song", "count")
    )
    .reset_index()
    .sort_values("year")
)

mode_summary = (
    df.groupby("mode")
    .agg(
        avg_popularity=("popularity", "mean"),
        avg_streams=("streams_millions", "mean"),
        song_count=("song", "count")
    )
    .reset_index()
)

corr_cols = [
    "streams_millions", "popularity", "energy", "danceability",
    "liveness", "valence", "duration", "acousticness", "speechiness", "bpm"
]

corr_table = df[corr_cols].corr().round(2)

display(top_songs)
display(top_artists)
display(genre_summary)
display(year_summary)
display(mode_summary)
display(corr_table)


## Step 5: Create charts

In [ ]:
output_dir = Path("spotify_site")
images_dir = output_dir / "images"

if output_dir.exists():
    shutil.rmtree(output_dir)

images_dir.mkdir(parents=True)

def save_barh(data, label_col, value_col, title, xlabel, filename):
    plt.figure(figsize=(8, 5))
    plt.barh(data[label_col].astype(str), data[value_col])
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlabel(xlabel)
    plt.tight_layout()
    path = images_dir / filename
    plt.savefig(path, dpi=160)
    plt.close()
    return "images/" + filename

def save_bar(data, label_col, value_col, title, ylabel, filename):
    plt.figure(figsize=(8, 5))
    plt.bar(data[label_col].astype(str), data[value_col])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    path = images_dir / filename
    plt.savefig(path, dpi=160)
    plt.close()
    return "images/" + filename

def save_scatter(x_col, y_col, title, xlabel, ylabel, filename):
    plt.figure(figsize=(7, 5))
    plt.scatter(df[x_col], df[y_col], alpha=0.75)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.35)
    plt.tight_layout()
    path = images_dir / filename
    plt.savefig(path, dpi=160)
    plt.close()
    return "images/" + filename

top_songs_chart = save_barh(top_songs, "song", "streams_millions", "Top 10 Songs by Spotify Streams", "Streams in Millions", "top_songs.png")
top_artists_chart = save_barh(top_artists, "artist", "streams_millions", "Top 10 Artists by Total Streams", "Streams in Millions", "top_artists.png")
genre_chart = save_barh(genre_summary, "genre", "avg_popularity", "Average Popularity by Genre", "Average Popularity", "genre_popularity.png")

plt.figure(figsize=(8, 5))
plt.plot(year_summary["year"], year_summary["total_streams"], marker="o")
plt.title("Total Streams by Year")
plt.xlabel("Year")
plt.ylabel("Total Streams in Millions")
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(images_dir / "streams_by_year.png", dpi=160)
plt.close()
year_chart = "images/streams_by_year.png"

dance_chart = save_scatter("danceability", "popularity", "Danceability vs. Popularity", "Danceability", "Popularity", "danceability_popularity.png")
energy_chart = save_scatter("energy", "popularity", "Energy vs. Popularity", "Energy", "Popularity", "energy_popularity.png")

corr_to_pop = corr_table["popularity"].drop("popularity").sort_values()
plt.figure(figsize=(8, 5))
plt.barh(corr_to_pop.index, corr_to_pop.values)
plt.axvline(0, linewidth=1)
plt.title("Correlation of Song Traits with Popularity")
plt.xlabel("Correlation")
plt.tight_layout()
plt.savefig(images_dir / "correlations.png", dpi=160)
plt.close()
corr_chart = "images/correlations.png"

mode_chart = save_bar(mode_summary, "mode", "avg_popularity", "Average Popularity by Mode", "Average Popularity", "mode_popularity.png")

print("Charts saved in:", images_dir)


## Step 6: Create the connected HTML files

In [ ]:
def table_html(dataframe, max_rows=10):
    show = dataframe.head(max_rows).copy()
    for col in show.columns:
        if pd.api.types.is_numeric_dtype(show[col]):
            show[col] = show[col].map(lambda x: f"{x:,.2f}")
    return show.to_html(index=False, border=0, classes="data-table")

style = """
<style>
  * { box-sizing: border-box; }
  body {
    margin: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #f4f6f5;
    color: #111827;
    line-height: 1.55;
  }
  header {
    background: #1DB954;
    color: white;
    padding: 70px 10%;
  }
  header h1 {
    margin: 0 0 20px;
    font-size: 44px;
    font-weight: 800;
  }
  header p {
    font-size: 20px;
    margin: 0 0 25px;
    font-weight: 600;
  }
  nav a {
    color: white;
    font-weight: 800;
    font-size: 18px;
    margin-right: 24px;
    text-decoration: underline;
  }
  main {
    max-width: 1120px;
    margin: 0 auto;
    background: white;
    padding: 56px 38px;
    min-height: 100vh;
  }
  .card {
    background: white;
    border-radius: 14px;
    padding: 28px;
    margin-bottom: 28px;
    box-shadow: 0 4px 18px rgba(15, 23, 42, 0.08);
    border: 1px solid #e5e7eb;
  }
  h2 {
    font-size: 30px;
    margin: 0 0 20px;
    font-weight: 800;
  }
  h3 {
    font-size: 22px;
    margin: 0 0 16px;
    font-weight: 800;
    border-bottom: 2px solid #1DB954;
    padding-bottom: 8px;
  }
  p, li { font-size: 18px; }
  .note {
    border-left: 8px solid #1DB954;
    background: #e9fbf1;
    padding: 20px 24px;
    border-radius: 12px;
    font-size: 18px;
  }
  .grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 18px;
    margin-top: 18px;
  }
  .stat {
    background: #ecfdf3;
    border-left: 5px solid #1DB954;
    border-radius: 10px;
    padding: 16px;
  }
  .stat strong {
    display: block;
    font-size: 28px;
    color: #14532d;
  }
  .button {
    display: inline-block;
    background: #1DB954;
    color: white;
    padding: 12px 18px;
    border-radius: 10px;
    font-weight: 800;
    text-decoration: none;
    margin-top: 8px;
  }
  .table-wrap {
    overflow-x: auto;
    margin-top: 12px;
  }
  .data-table {
    width: 100%;
    border-collapse: collapse;
    margin: 16px 0;
    font-size: 14px;
  }
  .data-table th {
    background: #1DB954;
    color: white;
    padding: 9px;
    text-align: left;
    border: 1px solid #16a34a;
  }
  .data-table td {
    padding: 8px;
    border: 1px solid #d7d7d7;
  }
  .chart-img {
    width: 100%;
    border: 1px solid #d7d7d7;
    border-radius: 10px;
    margin: 14px 0;
    background: white;
  }
  .insight {
    background: #e9fbf1;
    border-left: 5px solid #1DB954;
    border-radius: 8px;
    padding: 14px 16px;
    font-size: 16px;
    margin-top: 14px;
  }
  code {
    background: #f1f5f9;
    display: block;
    padding: 12px;
    border-radius: 8px;
    margin-top: 10px;
  }
  @media (max-width: 800px) {
    header { padding: 45px 7%; }
    header h1 { font-size: 34px; }
    main { padding: 32px 20px; }
    .grid { grid-template-columns: 1fr; }
    p, li { font-size: 16px; }
  }
</style>
"""

home_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Spotify Home Page</title>
  {style}
</head>
<body>
  <header>
    <h1>Spotify Songs Analysis</h1>
    <p>Python, Pandas, and Matplotlib analysis of a Spotify songs dataset.</p>
    <nav>
      <a href="spotify_home.html">Home</a>
      <a href="spotify_report.html">Analysis</a>
    </nav>
  </header>

  <main>
    <section class="card">
      <h2>Important Note</h2>
      <div class="note">
        This page is ready for GitHub Pages. The connected report page includes tables, charts, correlations, and final insights.
      </div>
    </section>

    <section class="card">
      <h2>Problem Space</h2>
      <p>
        This project studies what makes songs successful on Spotify. The analysis compares streaming totals with playlist placement,
        artist frequency, release timing, and audio features such as danceability, valence, energy, acousticness, liveness, and speechiness.
      </p>
    </section>

    <section class="card">
      <h2>Dataset Connection</h2>
      <p>
        The Spotify dataset is useful because it contains both popularity measures and music feature measures.
        This lets us ask whether highly streamed songs are popular because of their sound, their visibility, or both.
      </p>
    </section>

    <section class="card">
      <h2>Questions Addressed</h2>
      <ol>
        <li>Which songs have the most Spotify streams?</li>
        <li>Which artists appear most often in the dataset?</li>
        <li>Do songs in more Spotify playlists tend to have more streams?</li>
        <li>What audio features are most common among popular songs?</li>
        <li>Is danceability related to streams?</li>
        <li>Is energy related to streams?</li>
        <li>Do newer or older songs dominate the most-streamed list?</li>
        <li>Which keys and modes appear most often?</li>
      </ol>
      <a class="button" href="spotify_report.html">Go to Analysis</a>
    </section>

    <section class="card">
      <h2>Analysis Method</h2>
      <p>
        The notebook imports the dataset into a Pandas DataFrame, cleans numeric columns, creates new columns such as streams in millions,
        uses summary statistics, filters and groups the data, and creates visualizations with Matplotlib.
      </p>
    </section>

    <section class="card">
      <h2>Expected Insight Story</h2>
      <p>
        The project is designed to test whether audio features alone explain streaming success. The stronger expected story is that
        playlist visibility and platform reach matter a lot, while features like danceability and energy help describe songs but do not
        fully explain streams by themselves.
      </p>
    </section>
  </main>
</body>
</html>
"""

report_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Spotify Full Report</title>
  {style}
</head>
<body>
  <header>
    <h1>Spotify Songs Data Analysis</h1>
    <p>DS 220 Project 2: Python, pandas, NumPy, and Matplotlib</p>
    <nav>
      <a href="spotify_home.html">Home</a>
      <a href="spotify_report.html">Analysis</a>
    </nav>
  </header>

  <main>
    <section class="card">
      <h2>Problem Space</h2>
      <p>
        Music streaming changed how artists, labels, and listeners measure success. Instead of only looking at radio play or album sales,
        Spotify-style data lets us study streams, playlists, charts, and audio traits like energy, danceability, and valence.
      </p>
    </section>

    <section class="card">
      <h2>Dataset</h2>
      <p>
        The dataset contains popular Spotify songs from 2010 to 2019. It includes song, artist, genre, year, streams, playlist counts,
        chart appearances, BPM, mode, energy, danceability, valence, acousticness, speechiness, and popularity.
      </p>
      <code>{csv_file}</code>
    </section>

    <section class="card">
      <h2>Exploratory Data Analysis</h2>
      <p>The dataset was cleaned, checked, summarized, and converted into readable tables and charts.</p>
      <div class="table-wrap">{table_html(df.describe().T.reset_index().rename(columns={"index":"statistic"}), 12)}</div>
    </section>

    <section class="card">
      <h2>1. Which songs have the most streams?</h2>
      <div class="table-wrap">{table_html(top_songs, 10)}</div>
      <img class="chart-img" src="{top_songs_chart}" alt="Top songs chart">
      <div class="insight">The top-streamed songs are not always the newest songs. Replay value and playlist placement help songs stay relevant.</div>
    </section>

    <section class="card">
      <h2>2. Which artists have the highest total streams?</h2>
      <div class="table-wrap">{table_html(top_artists, 10)}</div>
      <img class="chart-img" src="{top_artists_chart}" alt="Top artists chart">
      <div class="insight">Artists with multiple major hits have an advantage because their total streams add across songs.</div>
    </section>

    <section class="card">
      <h2>3. Which genres have the highest average popularity?</h2>
      <div class="table-wrap">{table_html(genre_summary, 10)}</div>
      <img class="chart-img" src="{genre_chart}" alt="Genre chart">
      <div class="insight">Genres with fewer songs can rank highly if those songs are very successful.</div>
    </section>

    <section class="card">
      <h2>4. How do streams change by year?</h2>
      <div class="table-wrap">{table_html(year_summary, 20)}</div>
      <img class="chart-img" src="{year_chart}" alt="Year chart">
      <div class="insight">Yearly totals depend on how many major hits appear in the dataset for that year.</div>
    </section>

    <section class="card">
      <h2>5. Is danceability related to popularity?</h2>
      <p>Correlation: <strong>{round(df["danceability"].corr(df["popularity"]), 3)}</strong></p>
      <img class="chart-img" src="{dance_chart}" alt="Danceability scatterplot">
      <div class="insight">Danceability has some relationship with popularity, but it is not strong enough by itself to explain song success.</div>
    </section>

    <section class="card">
      <h2>6. Is energy related to popularity?</h2>
      <p>Correlation: <strong>{round(df["energy"].corr(df["popularity"]), 3)}</strong></p>
      <img class="chart-img" src="{energy_chart}" alt="Energy scatterplot">
      <div class="insight">High-energy songs can become major hits, but some lower-energy songs also perform very well.</div>
    </section>

    <section class="card">
      <h2>7. What traits are most correlated with popularity?</h2>
      <div class="table-wrap">{table_html(corr_table.reset_index().rename(columns={"index":"feature"}), 12)}</div>
      <img class="chart-img" src="{corr_chart}" alt="Correlation chart">
      <div class="insight">Popularity is connected to multiple traits rather than one simple factor.</div>
    </section>

    <section class="card">
      <h2>8. Do major-mode and minor-mode songs differ?</h2>
      <div class="table-wrap">{table_html(mode_summary, 10)}</div>
      <img class="chart-img" src="{mode_chart}" alt="Mode chart">
      <div class="insight">Mode may affect mood, but it is not a simple rule for predicting popularity.</div>
    </section>

    <section class="card">
      <h2>Final Insights</h2>
      <p>
        Spotify success is not explained by one feature alone. The highest-streamed songs usually combine broad pop appeal,
        playlist visibility, artist recognition, and strong replay value.
      </p>
    </section>
  </main>
</body>
</html>
"""

(output_dir / "spotify_home.html").write_text(home_html, encoding="utf-8")
(output_dir / "spotify_report.html").write_text(report_html, encoding="utf-8")

print("Created spotify_home.html and spotify_report.html")


## Step 7: Zip and download the website

In [ ]:
zip_name = "spotify_html_site.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for file in output_dir.rglob("*"):
        z.write(file, file.relative_to(output_dir))

files.download(zip_name)
print("Downloaded:", zip_name)
